# Chaining Cells

One of the Notebook's most powerful features is *cell chaining*: the ability to use the result of an earlier cell as input to a later one. This lets you build up a data pipeline step by step, inspecting what you have at each stage before deciding where to go next.

## How It Works

Check the **Use previous cells as context** checkbox that appears at the top of any cell after the first. When checked, each earlier cell's result is bound to a variable — `$cell1` for the first cell, `$cell2` for the second, and so on — and those variables are available in the current cell's query.

## Example: Exploring Shakespeare

The database contains Shakespeare's complete works. Let's explore it incrementally, building an analysis across three cells.

**Cell 1** — Collect all speeches across all plays. Run this first:

In [ ]:
collection("/db/apps/notebook/data/getting-started/data/shakespeare")//SPEECH

This returns the full sequence of speeches. It may be a large result — that's fine. Now instead of querying the database again, we pass the sequence forward.

**Cell 2** — Check **Use previous cells as context**, then count speeches per play. `$cell1` holds the result from above:

In [ ]:
for $speech in $cell1
group by $play := $speech/ancestor::PLAY/TITLE/string()
order by count($speech) descending
return $play || ": " || count($speech) || " speeches"

The `group by` clause partitions the speeches by play, so `count($speech)` inside the group gives the total per play. No second trip to the database.

**Cell 3** — Check **Use previous cells as context**, then drill into the most verbose play. `$cell2` tells us which play ranked first; `$cell1` gives us its speeches:

In [ ]:
(
    for $speech in $cell1[ancestor::PLAY/TITLE = substring-before($cell2[1], ":")]
    group by $speaker := string-join($speech/SPEAKER, " &amp; ")
    order by count($speech) descending
    return
        <speaker name="{$speaker}" speeches="{count($speech)}"/>
) => subsequence(1, 10)

The result is the ten characters who speak the most in Shakespeare's chattiest play.

## Version Declarations and Imports

You can include an `xquery version` declaration or `import module` statement in a chained cell — they are recognized as prolog declarations and hoisted correctly:

In [ ]:
xquery version "4.0";
$cell1
    => subsequence(1, 5)
    => for-each(fn { string(./SPEAKER) })
    => string-join(", ")

## Known Limitation: `declare function`

Module-level `declare function` and `declare variable` statements in the prolog cannot be used in chained cells, because the chaining mechanism injects `let` bindings between the prolog and the body, and parsing function-body braces reliably requires a full parser.

The workaround is to use **inline functions** in the query body instead:

In [ ]:
let $label := function($speech as element(SPEECH)) as xs:string {
    $speech/SPEAKER || " (" || count($speech/LINE) || " lines)"
}
for $speech in $cell1[position() = (1 to 5)]
return $label($speech)

If you need to share a helper function across several cells, store it as a library module in the database and `import module` it — that works fine.

## Why This Matters

The three cells form a natural pipeline — collect, aggregate, drill down — where each step builds on what came before. You could write this as a single query, but the cell-by-cell approach lets you:

- Inspect the shape of the data before committing to an aggregation
- Reuse an expensive database scan across multiple analytical steps
- Iterate on each step independently without re-running earlier ones

This mirrors how exploratory data analysis actually works: you discover what you have, then decide what to do with it.